In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
!ls /content/drive/MyDrive/Medicine-Delivery-Optimization

delivery_partners.csv  inventory.csv  notebooks  orders.csv


In [28]:
!pip install faker pandas numpy

In [29]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

In [30]:
NUM_ORDERS = 50000

cities = ["Bangalore", "Mumbai", "Delhi", "Chennai"]

warehouses = {
    "Bangalore": [101, 102, 103],
    "Mumbai": [201, 202],
    "Delhi": [301, 302],
    "Chennai": [401, 402]
}

vehicle_types = ["bike", "cycle"]
medicine_types = ["chronic", "acute"]

In [31]:
partners = []

for i in range(1, 201):
    partners.append({
        "partner_id": i,
        "avg_rating": round(np.random.uniform(2.5, 5.0), 2),
        "vehicle_type": random.choice(vehicle_types)
    })

partners_df = pd.DataFrame(partners)
partners_df.to_csv("/content/drive/MyDrive/Medicine-Delivery-Optimization/delivery_partners.csv", index=False)

partners_df.head()

,partner_id,avg_rating,vehicle_type
0,1,3.44,bike
1,2,4.88,bike
2,3,4.33,cycle
3,4,4.00,bike
4,5,2.89,bike


In [32]:
inventory = []

for wh_list in warehouses.values():
    for wh in wh_list:
        for med_id in range(1, 21):
            inventory.append({
                "warehouse_id": wh,
                "medicine_id": med_id,
                "stock_available": np.random.choice(
                    [0, 10, 20, 50, 100],
                    p=[0.05, 0.25, 0.3, 0.3, 0.1]  # 5% stock-out
                )
            })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv("/content/drive/MyDrive/Medicine-Delivery-Optimization/inventory.csv", index=False)

inventory_df.head()

,warehouse_id,medicine_id,stock_available
0,101,1,50
1,101,2,10
2,101,3,10
3,101,4,50
4,101,5,50


In [44]:
orders = []

start_date = datetime(2024, 1, 1)

for i in range(NUM_ORDERS):

    order_id = i + 1
    city = random.choice(cities)
    warehouse_id = random.choice(warehouses[city])
    partner = random.choice(partners)

    medicine_type = random.choice(medicine_types)
    order_value = round(np.random.uniform(100, 2000), 2)

    # -------------------------------
    # ⏰ ORDER TIME (controlled distribution)
    # -------------------------------
    if random.random() < 0.25:
        hour = random.randint(0, 5)   # late night
    else:
        hour = random.randint(6, 23)  # normal hours

    order_time = start_date + timedelta(
        days=random.randint(0, 90),
        hours=hour,
        minutes=random.randint(0, 59)
    )

    # -------------------------------
    # 📦 STOCK CHECK
    # -------------------------------
    stock = inventory_df[
        inventory_df["warehouse_id"] == warehouse_id
    ].sample(1)["stock_available"].values[0]

    # -------------------------------
    # 🚚 DISPATCH DELAY
    # -------------------------------
    dispatch_delay = random.uniform(0.5, 1.5)

    # Stock-out impact (stronger)
    if stock == 0 and random.random() < 0.8:
        dispatch_delay += random.uniform(1.5, 3)

    # Late-night impact (stronger)
    if (order_time.hour >= 22 or order_time.hour <= 5) and random.random() < 0.6:
        dispatch_delay += random.uniform(0.8, 1.8)

    dispatch_time = order_time + timedelta(hours=dispatch_delay)

    # -------------------------------
    # 🚴 DELIVERY DELAY
    # -------------------------------
    delivery_delay = random.uniform(2.2, 4.5)

    # Low rating impact
    if partner["avg_rating"] < 3 and random.random() < 0.7:
        delivery_delay += random.uniform(0.5, 1.2)

    # City traffic (stronger)
    if city == "Mumbai" and random.random() < 0.7:
        delivery_delay += random.uniform(0.7, 1.5)

    if city == "Bangalore" and random.random() < 0.6:
        delivery_delay += random.uniform(0.5, 1.2)

    delivery_time = dispatch_time + timedelta(hours=delivery_delay)

    # -------------------------------
    # 📊 SLA BREACH
    # -------------------------------
    total_time_hours = (delivery_time - order_time).total_seconds() / 3600
    sla_breach = total_time_hours > 6

    orders.append({
        "order_id": order_id,
        "city": city,
        "order_time": order_time,
        "dispatch_time": dispatch_time,
        "delivery_time": delivery_time,
        "warehouse_id": warehouse_id,
        "partner_id": partner["partner_id"],
        "medicine_type": medicine_type,
        "order_value": order_value,
        "sla_breach": sla_breach
    })

orders_df = pd.DataFrame(orders)
orders_df.head()

,order_id,city,order_time,dispatch_time,delivery_time,warehouse_id,partner_id,medicine_type,order_value,sla_breach
0,1,Mumbai,2024-01-24 06:16:00,2024-01-24 06:46:35.276951,2024-01-24 13:02:05.570877,202,30,acute,818.80,True
1,2,Chennai,2024-02-20 06:49:00,2024-02-20 08:05:11.295583,2024-02-20 11:05:35.248481,401,26,chronic,749.75,False
2,3,Bangalore,2024-01-28 14:28:00,2024-01-28 15:00:23.162199,2024-01-28 19:38:03.538299,102,22,chronic,469.43,False
3,4,Delhi,2024-02-16 06:08:00,2024-02-16 06:45:40.231684,2024-02-16 10:22:20.845151,301,117,acute,1880.74,False
4,5,Chennai,2024-02-06 18:30:00,2024-02-06 19:57:57.426540,2024-02-06 23:13:49.212603,402,150,acute,1014.30,False


In [45]:
orders_df.to_csv("/content/drive/MyDrive/Medicine-Delivery-Optimization/orders.csv", index=False)

In [46]:
print("Total Orders:", len(orders_df))

print("\nSLA Breach Rate:")
print(orders_df["sla_breach"].mean() * 100)

print("\nCity Distribution:")
print(orders_df["city"].value_counts())

print("\nLate Night Orders:")
late_night = orders_df[
    (orders_df["order_time"].dt.hour >= 22) |
    (orders_df["order_time"].dt.hour <= 5)
]
print(len(late_night))

Total Orders: 50000

SLA Breach Rate:
23.508000000000003

City Distribution:
city
Delhi        12760
Chennai      12611
Mumbai       12362
Bangalore    12267
Name: count, dtype: int64

Late Night Orders:
16534


In [47]:
from google.colab import files

files.download("/content/drive/MyDrive/Medicine-Delivery-Optimization/orders.csv")
files.download("/content/drive/MyDrive/Medicine-Delivery-Optimization/inventory.csv")
files.download("/content/drive/MyDrive/Medicine-Delivery-Optimization/delivery_partners.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>